In [3]:
import pandas as pd
import numpy as np
import duckdb

con = duckdb.connect('model.duckdb')


In [7]:
#load hierarchies
shtNames = pd.ExcelFile('hierarchies/hy_specs.xlsx').sheet_names
shtNames.remove('HY_Template')
print(shtNames)
for sht in shtNames:
    print(sht)
    res_exl = pd.read_excel('hierarchies/hy_specs.xlsx',sheet_name=sht, header=0, index_col=[0,1], dtype={'id':np.intc, 'parent_id':np.intc, 'name':np.str_, 'unit':np.str_, 'parent_name':np.str_, 'level_number':np.intc, 'is_root':np.bool_, 'is_leaf':np.bool_,'weight':np.intc, 'is_calculated':np.bool_, 'natural_key':np.str_, 'level_name':np.str_}, usecols='A:L')
    res_exl.to_csv('hierarchies/'+str(sht)+'.csv')
    sql = '''
    CREATE OR REPLACE TABLE ''' + str(sht) + '''
    (
    id                INTEGER,
    parent_id         INTEGER,  
    name              VARCHAR(100) NOT NULL,
    unit              VARCHAR(100),  
    parent_name       VARCHAR(100),
    level_number      INTEGER,
    is_root           BOOLEAN DEFAULT FALSE,
    is_leaf           BOOLEAN DEFAULT TRUE,
    weight            INTEGER,
    is_calculated     BOOLEAN DEFAULT FALSE,
    natural_key       INTEGER,
    level_name        VARCHAR(100),
    CONSTRAINT '''+str(sht)+'''_hy_pk PRIMARY KEY (id), 
    CONSTRAINT '''+str(sht)+'''_hy_uk_1 UNIQUE (natural_key), 
    CONSTRAINT '''+str(sht)+'''_hy_self_fk FOREIGN KEY (parent_id)
        REFERENCES '''+str(sht)+''' (id)
    )
    '''

    con.sql(sql)

    
    sql = '''
    SELECT *  
    FROM read_csv('hierarchies/'''+str(sht)+'''.csv', header = True,
    columns = {
    'id':                'INTEGER',
    'parent_id':         'INTEGER',  
    'name':              'VARCHAR',
    'unit':              'VARCHAR',
    'parent_name':       'VARCHAR',   
    'level_number':      'INTEGER',
    'is_root':           'BOOLEAN',
    'is_leaf':           'BOOLEAN',
    'weight':            'INTEGER',
    'is_calculated':     'BOOLEAN',
    'natural_key':       'INTEGER',   
    'level_name':        'VARCHAR'
    }    
    )        
    '''

    res = con.sql(sql)
    df7 = pd.DataFrame(res.fetchall(), columns = ['id', 'parent_id','name','unit','parent_name','level_number','is_root','is_leaf','weght','is_calculated','natural_key','level_name'])
    df7['parent_id'].replace(to_replace=0, value=pd.NA, inplace=True)
    df_list = df7.values.tolist()
    con.executemany('INSERT INTO '+str(sht)+' VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)', df_list)

    # Insert Dataframe into SQL Server:
    sql = 'SELECT * FROM '+str(sht)
    con.sql(sql)





['Years', 'Versions', 'Months', 'Dispatch_Plants', 'Production_Plants', 'Cement_types', 'Cements_mills', 'Cement_mill_KPIs', 'Cement_mill_norms', 'Paccing_types', 'Recipes_raw_meal_indicators', 'Recipes_cement_indicators']


ConstraintException: Constraint Error: Duplicate key "id: 2" violates primary key constraint

IndentationError: unexpected indent (1488601320.py, line 21)


    id                INTEGER,
    parent_id         INTEGER,  
   
    name              VARCHAR,
    weight            INTEGER,
    level_number      INTEGER,
    level_name        VARCHAR,
    parent_name       VARCHAR,
    unit              VARCHAR,
    is_root           BOOLEAN,
    is_leaf           BOOLEAN,
    is_calculated     BOOLEAN,
    natural_key       INTEGER

In [ ]:
con.close()


In [ ]:
duckdb.sql("PRAGMA show_tables;")

┌─────────────┐
│    name     │
│   varchar   │
├─────────────┤
│ _migrations │
└─────────────┘